## Import Deps

In [ ]:
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from src.utils.preprocess import split_train_test, preprocess_pipe

## Load Dataset and split between train and test

In [ ]:
ds_path = "https://simplonline-v3-prod.s3.eu-west-3.amazonaws.com/media/file/csv/dataset-electric-6b0361bc9dd9f506891054.csv"
dataset = preprocess_pipe(ds_path)
X_train, X_test, y_train, y_test = split_train_test(dataset)

## Now we fit XGboost Regresser to our dataset

In [ ]:
preprocessor = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore"), ["Hour", "DayOfWeek", "Month"])],
    remainder="passthrough"
)

pipe = Pipeline([
    ("prep", preprocessor),
    ("model", XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, n_jobs=-1))
])

pipe.fit(X_train, y_train)

## Model Evaluation

In [ ]:
pred = pipe.predict(X_test)
print("MAE :", mean_absolute_error(y_test, pred))
print("RMSE:", root_mean_squared_error(y_test, pred))

baseline = X_test["lag24"]
print("Baseline MAE :", mean_absolute_error(y_test, baseline))
print("Baseline RMSE:", root_mean_squared_error(y_test, baseline))

In [ ]:
from sklearn.metrics import r2_score

print("R2      :", r2_score(y_test, pred))
print("Baseline:", r2_score(y_test, baseline))